Exercise 9: Build an LSTM Model for Sentiment Classification


Aim
To build and evaluate a Long Short-Term Memory (LSTM) model for classifying text sentiment into Positive and Negative classes using deep learning.

In [1]:
pip install tensorflow scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.6/245.6 MB 4.5 MB/s  0:00:46 eta 0:00:010:00:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 3.8 MB/s  0:00:033.9 MB/s eta 0:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 4.1 MB/s  0:00:004.2 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 562.6/562.6 kB 3.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 3.1 MB/s  0:00:00m 3.4 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 3.1 MB/s  0:00:08m 3.1 MB/s eta 0:00:01
  Attempting uninstall: h5py━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━  6/15 [ml_dtypes]ree]g]
    Found existing installation: h5py 3.16.0237m╺━━━━━━━━━━━━━━━━━━━━━━━  6/15 [ml_dtypes]
    Uninstalling h5py-3.16.0:━━━━╺━━━━━━━━━━━━━━━━━━━━━━━  6/15 [ml_dtypes]
      Successfully uninstalled h5py-3.16.05;237m╺━━━━━━━━━━━━━━━━━━━━━━━  6/15 [ml_dtypes]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15/15 [tensorflow]0m 14/15 [tensorflow]s-nightly]
Note: 

In [ ]:

import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix


texts = [
    "I love this product",
    "This is an excellent product",
    "The movie was wonderful",
    "I am very happy",
    "Amazing quality and service",
    "I really enjoyed this movie",
    "This product is fantastic",
    "The experience was great",
    "I hate this product",
    "This is a terrible product",
    "The movie was boring",
    "I am very sad",
    "Very poor quality and service",
    "I really disliked this movie",
    "This product is awful",
    "The experience was bad",
    "I love this movie",
    "The service is excellent",
    "I hate this movie",
    "The service is terrible"
]

labels = np.array([
    1, 1, 1, 1, 1, 1, 1, 1, 0, 0,
    0, 0, 0, 0, 0, 0, 1, 1, 0, 0
])


tokenizer = Tokenizer()
tokenizer.fit_on_texts(texts)

sequences = tokenizer.texts_to_sequences(texts)
max_length = 6
X = pad_sequences(sequences, maxlen=max_length, padding="post")

vocab_size = len(tokenizer.word_index) + 1

X_train, X_test, y_train, y_test = train_test_split(
    X, labels, test_size=0.25,
    random_state=42, stratify=labels
)


model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=16),
    LSTM(16),
    Dense(1, activation="sigmoid")
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.build(input_shape=(None, max_length))
model.summary()

# Step 5: Train the model
model.fit(
    X_train, y_train,
    epochs=15,
    batch_size=4,
    verbose=1
)


predictions = model.predict(X_test, verbose=0)
y_pred = (predictions >= 0.5).astype(int).flatten()

print("\nTest Accuracy:",
      round(accuracy_score(y_test, y_pred), 2))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))


new_texts = [
    "I really love this product",
    "This product is very bad"
]

new_sequences = tokenizer.texts_to_sequences(new_texts)
new_padded = pad_sequences(
    new_sequences, maxlen=max_length, padding="post"
)

new_predictions = model.predict(new_padded, verbose=0)

print("\nSentiment Predictions:")
for text, score in zip(new_texts, new_predictions.flatten()):
    sentiment = "Positive" if score >= 0.5 else "Negative"
    print(text, "->", sentiment)
